# Scipy Feature Tracking Algorithm

### Import Packages

In [ ]:
import xarray as xr
from scipy.ndimage import label, generate_binary_structure
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import pandas as pd

## Let's go through the basic steps to identify each precipitation feature.

Open the dataset using xarray

In [ ]:
data = xr.open_dataset('data/precip_ex.nc')

Select a time to specifically do the subset of the data

In [ ]:
time = data.time[0]
subset_data = data.sel(time=time)

In order to account for individual features, anywhere less than 0.25mm/hr of precip will be set to zero. This is so areas of minimal/weak precipitation are not grouped together with zero to create one mega precip field over the entire Earth.

In [ ]:
no_minimal_precip = subset_data['ETC_tp'].where(subset_data['ETC_tp'] > 0.25, 0)
s = generate_binary_structure(2,2)

Run through labeling feature algorithm

In [ ]:
labeled_data, num_data = label(no_minimal_precip, structure=s)

## Let's look at all of the data plotted first.

In [ ]:
plt.figure(figsize=(15,8))
ax = plt.axes(projection=ccrs.PlateCarree())

## Create two-dimensional grid for latitude and longitude
lon2d, lat2d = np.meshgrid(no_minimal_precip['longitude'], no_minimal_precip['latitude'])

plt.contourf(lon2d, lat2d, labeled_data, cmap='tab20', transform=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.BORDERS, edgecolor='black')
ax.add_feature(cfeature.STATES, edgecolor='black')

ax.add_feature(cfeature.COASTLINE, edgecolor='black')

provinces = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_1_states_provinces_lines',
    scale='50m',
    facecolor='none',
    edgecolor='black'
)
ax.add_feature(provinces)
ax.gridlines(draw_labels=True)

Woah! This is neat! But how does this algorithm truly work? To understand this, let's focus on one particular precipitation event. 

## Understanding the Algorithm: Plotting One Specific Feature Through Time

Let's pull out the subset of the data that is specific to a partiuclar time.

In [ ]:
def precip_time_index(full_data, time_index):
    time = full_data.time[time_index]
    subset_data = full_data.sel(time=time)

    return subset_data

Let's create a function that will perform the algorithm.

In [ ]:
def precip_feature(data, time_index):
    subset_data = precip_time_index(data, time_index)

    no_minimal_precip = subset_data['ETC_tp'].where(subset_data['ETC_tp'] > 0.25, 0)
    s = generate_binary_structure(2,2)

    labeled_data, num_data = label(no_minimal_precip, structure=s)

    return labeled_data

If we only want to identify one precipitation feature, let's choose the feature that is associated with id == 1. We'll create a function to create this subsetted data.

In [ ]:
def subset_precip_feature(full_data):
    return np.where(full_data, full_data==1, 0)

Let's go ahead and plot this one specific feature throughout the full 12 hours of the event.

In [ ]:
for i in range(13):
    plt.figure(figsize=(15,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    lon2d, lat2d = np.meshgrid(no_minimal_precip['longitude'], no_minimal_precip['latitude'])
    subset_labeled_data = subset_precip_feature(precip_feature(data, i))
    plt.contour(lon2d, lat2d, subset_labeled_data, cmap='tab20b', transform=ccrs.PlateCarree())
    precip_data = precip_time_index(data, i)['ETC_tp'].to_numpy()
    precip_data[precip_data <= 0] = np.nan
    im = plt.pcolormesh(lon2d, lat2d, precip_data, cmap='BrBG', transform=ccrs.PlateCarree(), vmin=0, vmax=1)

    plt.colorbar(im, label='Precipitation Rate [mm/hr]', shrink=0.4)

    ax.set_extent([60, 150, 70, 80])

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.BORDERS, edgecolor='black')
    ax.add_feature(cfeature.STATES, edgecolor='black')
    
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')

    plt.title('Precip Feature 1 at Time Index ' + str(i))
    
    provinces = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none',
        edgecolor='black'
    )
    ax.add_feature(provinces)
    ax.gridlines(draw_labels=True)

These plots show the identification of a single precipitation feature throughout time (contoured). The filled contours are the hourly precipitation rate. The dark brown areas throughout each map represent the area where no precipitation was recorded. Therefore, each area that isn't dark brown area represents a location that received some precipitation. 

At time index 0, we can see that feature one corresponds to a large area of precipitation between 95°E to 115°E, as made out by the purple contour. As this area is continuously above 0.25mm/hr, the entire area is assigned to a single feature. With increasing time, the feature can be seen moving eastward, eventually reaching 125°E at time index 6. 

However, at time index 7, the contour no longer corresponds to the area of precipitation centered over 120°E. Instead, the feature is centered around 100°E. Between time index 6 and 7, these two precipitation areas break apart. In between these two precipitation features, there is now an area that has a precipitation rate of less than 0.25mm/hr, the critical rate we set for determining whether it is "precipitating" or not. Since these two features can no longer be connected, they are now treated as seperate features. Therefore, the purple contour is located only over the precipitation feature at 100°E and not 120°E. 

Between time indicies 7-9, we can observe the precipitation feature continuously shrinks, eventually reaching zero precipitation at time index 10. Since there is no precipitation features within this area, the index assigned to feature 1 is now shifted to a different location, and is no longer associated with the feature over northern Siberia. 

## Let's create an animation of the evolution of this single precipitation feature throughout time. 

In [ ]:
## Let's create a function to write the figures necessary for the animation.

## Creating a colorbar across all frames.
vmin = 0
vmax = 1
levels = np.linspace(vmin, vmax, 10)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap='BrBG'

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, shrink=0.4)
cbar.set_label("Precipitation Rate [mm/hr]")

def func(i):
    ax.clear()
    lon2d, lat2d = np.meshgrid(no_minimal_precip['longitude'], no_minimal_precip['latitude'])
    subset_labeled_data = subset_precip_feature(precip_feature(data, i))
    plt.contour(lon2d, lat2d, subset_labeled_data, cmap='tab20b', transform=ccrs.PlateCarree())
    precip_data = precip_time_index(data, i)['ETC_tp'].to_numpy()
    precip_data[precip_data <= 0] = np.nan
    im = plt.pcolormesh(lon2d, lat2d, precip_data, cmap='BrBG', transform=ccrs.PlateCarree(), vmin=0, vmax=1)

    ax.set_extent([60, 150, 70, 80])

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.BORDERS, edgecolor='black')
    ax.add_feature(cfeature.STATES, edgecolor='black')
    
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')

    ax.set_title('Precip Feature 1 at Time Index ' + str(i))
    
    provinces = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none',
        edgecolor='black'
    )
    ax.add_feature(provinces)
    ax.gridlines(draw_labels=True)
    return fig,


anim = FuncAnimation(fig, func, frames=12, interval=300, blit=False)
HTML(anim.to_jshtml())
# anim.save('my_animation.gif', writer='pillow', fps=2)

## Now let's apply this method to worldwide cases.

In [ ]:
for i in range(13):
    plt.figure(figsize=(15,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    lon2d, lat2d = np.meshgrid(no_minimal_precip['longitude'], no_minimal_precip['latitude'])
    labeled_data = precip_feature(data, i)
    plt.contour(lon2d, lat2d, labeled_data, cmap='tab20b', transform=ccrs.PlateCarree())
    precip_data = precip_time_index(data, i)['ETC_tp'].to_numpy()
    precip_data[precip_data <= 0] = np.nan
    im = plt.pcolormesh(lon2d, lat2d, precip_data, cmap='BrBG', transform=ccrs.PlateCarree(), vmin=0, vmax=1)

    plt.colorbar(im, label='Precipitation Rate [mm/hr]', shrink=0.4)

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.BORDERS, edgecolor='black')
    ax.add_feature(cfeature.STATES, edgecolor='black')
    
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')

    plt.title('Precip Features at Time Index ' + str(i))
    
    provinces = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none',
        edgecolor='black'
    )
    ax.add_feature(provinces)
    ax.gridlines(draw_labels=True)

Ok, so we can't really see much from this diagram, since there's a lot of countering going on... Instead of contouring based upon label number, let's contour based upon the maximum precipitation rate found within the countered region. 

## Working with Maximum Precipitation Rate Found within Countered Region

Create an empty array to hold the maximum precipitation rate. 

In [ ]:
max_precip_rate = np.empty(0)

Get data for time index 0.

In [ ]:
lon2d, lat2d = np.meshgrid(no_minimal_precip['longitude'], no_minimal_precip['latitude'])
labeled_data = precip_feature(data, 0)

Get the total number of features within the dataset.

In [ ]:
num_features = labeled_data.max()

In [ ]:
precip_time_index(data, 0)['ETC_tp'].to_numpy()[labeled_data == i]

Go through each feature to determine the maximum precipitation rate within each feature.

In [ ]:
## Create a copy of labeled dataset.
new_labeled_data = np.copy(labeled_data)
for i in range(1, num_features):
    max_precip_rate = np.append(max_precip_rate, precip_time_index(data, 0)['ETC_tp'].to_numpy()[labeled_data == i].max())
    ## In labeled data, assign every value within labeled_data to max_precip rate assigned. 
    new_labeled_data[labeled_data == i]
    
    

In [ ]:
max_precip_rate = np.append(max_precip_rate, precip_time_index(data, 0)['ETC_tp'].to_numpy()[labeled_data == i].max())
## In labeled data, assign every value within labeled_data to max_precip rate assigned. 
new_labeled_data = np.where(new_labeled_data, labeled_data != i, max_precip_rate[-1])

In [ ]:
new_labeled_data.max()

In [ ]:
plt.figure(figsize=(15,8))
ax = plt.axes(projection=ccrs.PlateCarree())
lon2d, lat2d = np.meshgrid(no_minimal_precip['longitude'], no_minimal_precip['latitude'])
subset_labeled_data = subset_precip_feature(precip_feature(data, 0))
im = plt.contour(lon2d, lat2d, new_labeled_data, cmap='BrBG', transform=ccrs.PlateCarree(), vmin=0, vmax=1)

plt.colorbar(im, label='Maximum Precipitation Rate [mm/hr]')